# Loss functions

The loss function is a fundamental component of the RN training process. It measures the distance between the output obtained by the model (for a given input) and the desired output. This value is used by the model optimizer to calculate the gradients of the model, at the end of each step of the training process.

There are two categories of loss functions:

- Regression: They are useful for optimization processes whose objective is to predict a numerical value in a continuous interval (e.g. the probability of rain).
- Classification: They are useful for tasks in which the objective is to predict a value among several possible categorical values ​​(e.g. the species of an animal that appears in a photograph).

In PyTorch, there are implementations for several popular optimization functions. Before looking at some examples, let's see how we can perform the calculation of a simple loss function using basic operations on tensors.




In [1]:
import torch

# Generates two synthetic neural network outputs
output1 = torch.tensor([1.0, 2.0, 3.0, 4.0])  # Sample output 1
output2 = torch.tensor([0.5, 1.5, 2.5, 3.5])  # Sample output 2

# Generates a vector of target values ​​(ground truth)
target = torch.tensor([0.8, 1.7, 2.9, 3.8])

# Calculates the squared difference between the outputs and the target values
squared_diff1 = torch.pow(output1 - target, 2)
squared_diff2 = torch.pow(output2 - target, 2)

# Calculates the mean square error (MSE) between the outputs and the target values
loss = torch.mean(squared_diff1) + torch.mean(squared_diff2)
print("MSE Loss:", loss.item())


MSE Loss: 0.14000000059604645


There is a function in Pytorch that implements MSE, which can be used as follows:

In [2]:
import torch
output1 = torch.tensor([1.0, 2.0, 3.0, 4.0])  # Sample output 1
output2 = torch.tensor([0.5, 1.5, 2.5, 3.5])  # Sample output 2
target = torch.tensor([0.8, 1.7, 2.9, 3.8])

# Calculates the mean square error (MSE) between the outputs and the target values ​​using the MSELoss function
criterion = torch.nn.MSELoss()
loss = criterion(output1, target) + criterion(output2, target)
print("MSE Loss:", loss.item())


MSE Loss: 0.14000000059604645


From now on we will use examples that always use functions that implement popular loss function calculations in Pytorch.

Moving on, within the category of regressive loss functions, we now see an example, based on the same use case of the L1Loss loss function.

In [3]:
# Calculates the root mean square (L1) error between the outputs and the target values ​​using the L1Loss function
criterion = torch.nn.L1Loss()
loss = criterion(output1, target) + criterion(output2, target)
print("L1 Loss:", loss.item())

L1 Loss: 0.5


We see that in this case the loss value obtained is much higher. The difference is that MSELoss used the square of the differences, while L1Loss uses the absolute value of the difference for its calculation.

Let's now look at a reproduction of the same example, using the Mean Bias Error (MBE) regressive loss function. In this case, there is no implementation for Pytorch, so we use a home-made implementation.

In [4]:
# Calculates the Mean Bias Error (MBE) between the outputs and the target values
diff1 = output1 - target
diff2 = output2 - target
print(diff1)
print(diff2)
loss = torch.mean(diff1) + torch.mean(diff2)
print("MBE Loss:", loss.item()/2)

tensor([0.2000, 0.3000, 0.1000, 0.2000])
tensor([-0.3000, -0.2000, -0.4000, -0.3000])
MBE Loss: -0.05000001937150955


# Activity

Given the above actual and desired outputs: *outpu1*, *output2*, *target*, calculate a loss function with the following formulation.

FPP = ∑ |output1 - target| + ∑ |output2 - target|


In [5]:
# Put your code here

## Categorical loss functions

Let us now explore several examples of categorical loss functions. In this case, we are addressing problems where the output of the model is a vector of probabilities. Each element of this vector measures the probability that the input is classified into the corresponding category.

A simple example would be a RN for which the number contained in images from the CIFAR-10 dataset is predicted. The output is a vector of 10 elements, where the first element contains the probability that the number in the image is a 0, the second element the probability that it is a 1, etc.

Loss functions of this type must capture the difference between the predicted class and the probabilities contained in the output vector obtained from the model inference. The predicted class assigns 100% probability to the correct class and 0 to all others, therefore loss functions must capture how far the model probabilities deviate from this perfect prediction.

Categorical loss functions try to capture this difference. One of them is *SVM Loss* and we now see an example of this function whose simplified formula is SVM Loss = ∑ max(0, 1 - y_i * f(x_i))


In [6]:
import torch

# Example prediction and target
predictions = torch.tensor([0.85, 0.12, 0.08, 0.01, 0, 0, 0.23, 0, 0.4 ,0.2]) 
targets = torch.tensor([1, 0, 0, 0, 0, 0, 0, 0, 0 ,0])

# SVM loss calculation
loss = torch.mean(torch.max(torch.zeros_like(targets), 1 - targets * predictions))
print("SVM Loss:", loss.item())


SVM Loss: 0.9149999618530273


The Pytorch function *MultiLabelMaginLoss* provides an implementation of this metric in Pytorch. Let's see the same example using this function.

In [7]:
# SVM loss calculation with Pytorch
loss = torch.nn.MultiLabelMarginLoss()
loss = loss(predictions.unsqueeze(0), targets.unsqueeze(0))
print("SVM Loss:", loss.item())

SVM Loss: 2.7039999961853027


Another example of a categorical loss function is Cross Entropy Likelihood Loss. It is one of the most popular and is implemented in Pytorch with the *torch.nn.CrossEntropyLoss* function. Let's see the same example, using this function.


In [8]:
# Calculation of CEL loss
loss = torch.nn.CrossEntropyLoss()
loss = loss(predictions.unsqueeze(0), targets.argmax().unsqueeze(0))
print("CEL Loss:", loss.item())

CEL Loss: 1.6783099174499512


# Exercise

Review the contents of the code below and try alternately using loss_function1, loss_function2, and loss_function3 as loss functions.

To do this, you have to change the line of code

*loss = loss_func1(outputs, labels)*

for the other version
and adjust the way the output layer is processed in the *forward* pass. See commented code.

Do you see any differences in their effectiveness?

In [ ]:
# Import libraries
import torch
import torchvision
import torchvision.transforms as transforms

# Download and prepare the CIFAR-10 dataset
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck')

# Create a neural network model
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # Output layer when using CrossEntropyLoss
        x = self.fc3(x)
        # Output layer when using NLLLoss
        # x=F.log_softmax(self.fc3(x), dim=1)
        return x

# Initialize the model
net = Net()
# Define two loss functions
loss_func1 = nn.CrossEntropyLoss()
loss_func2 = nn.NLLLoss()

# Define an optimizer
import torch.optim as optim
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# Training the network using CrossEntropyLoss
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = loss_func1(outputs, labels)
        # loss = loss_func2(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

print('Finished Training')


Files already downloaded and verified
Files already downloaded and verified
[1,  2000] loss: 2.241
